In [ ]:
from textgrid import TextGrid
import os
import pandas as pd

import statistics
from collections import defaultdict
from collections import Counter
import csv
import re

In [ ]:
df = pd.read_csv("...", sep="\t")
print(df.head())

In [ ]:
df2 = df[["ID", "gender", "birth_year", "ankeedi_kp", "education"]]
df2["ankeedi_year"] = df2["ankeedi_kp"].apply(
    lambda x: x.split('.')[2].split('/')[0]
)
df2["age"] = df2["ankeedi_year"].astype(int) - df2["birth_year"].astype(int)
print(df2.head())

In [ ]:
df_final = df2[["ID", "gender", "ankeedi_year", "age", "education"]]
print(df_final)
df_final.to_csv('speakers.csv')

In [ ]:
file_paths = ['...']

rows = []
sounds = []

# 1) precompute: fast access to participant metadata
info_map = (df_final
            .set_index("ID")[["ankeedi_year", "age", "education"]]
            .to_dict("index"))

def overlaps(a_min, a_max, b_min, b_max):
    return (b_min < a_max) and (b_max > a_min)

def collect_morphs_for_words(words_tier, morph_tier):
    """
    2) fast collection of morphs for each word in a single pass (two pointers),
    instead of W*M.
    Returns a dict: word_index -> list_of_morph_marks
    """
    word_to_morphs = {i: [] for i, w in enumerate(words_tier) if w.mark.strip()}
    word_items = [(i, w.minTime, w.maxTime) for i, w in enumerate(words_tier) if w.mark.strip()]

    # Just in case, sort (usually already sorted)
    word_items.sort(key=lambda x: x[1])
    morph_items = [(m.minTime, m.maxTime, m.mark.strip()) for m in morph_tier if m.mark.strip()]
    morph_items.sort(key=lambda x: x[0])

    j = 0
    for i, w_min, w_max in word_items:
        # advance morphs that definitely end before the word
        while j < len(morph_items) and morph_items[j][1] <= w_min:
            j += 1

        k = j
        # collect all morphs that may overlap with the word
        while k < len(morph_items) and morph_items[k][0] < w_max:
            m_min, m_max, m_mark = morph_items[k]
            if overlaps(w_min, w_max, m_min, m_max):
                word_to_morphs[i].append(m_mark)
            k += 1

    return word_to_morphs

n_fp = len(file_paths)
for i_fp, file_path in enumerate(file_paths):
    print(f"Processing {i_fp + 1}/{n_fp} file path ({(i_fp + 1) / n_fp * 100:.1f}%)")

    condition = 0 if file_path == 'SKK0_TG' else 1

    files = [f for f in os.listdir(file_path) if f.endswith(".TextGrid")]
    n_f = len(files)

    for i_f, filename in enumerate(files):
        print(f"Processing {i_f + 1}/{n_f} file ({(i_f + 1) / n_f * 100:.1f}%)")
        file = os.path.join(file_path, filename)

        try:
            tg = TextGrid.fromFile(file)
        except Exception as e:
            print("Failed:", file, e)
            continue

        base = filename.replace(".TextGrid", "")
        left, gender_letter = base.split("_")
        
        # Separate dialogue number and participant number
        dialogue_id, id_number = left.removeprefix("SKK").split("-", 1)
        
        participant_id = f"{id_number}_{gender_letter}"
        gender = 0 if gender_letter == "N" else 1
        info = info_map.get(participant_id)
        if info is None:
            # if the participant is missing in df_final — skip or set None
            continue
        year = info["ankeedi_year"]
        age = info["age"]
        education = info["education"]

        words_tier = next(t for t in tg.tiers if t.name == "sõnad")
        morph_tier = next(t for t in tg.tiers if t.name == "morf")

        # 3) collect morphs for all words ONCE
        word_to_morphs = collect_morphs_for_words(words_tier, morph_tier)

        # 4) real_word_indices — also computed once
        real_word_indices = [i for i, w in enumerate(words_tier)
                             if w.mark.strip() and word_to_morphs.get(i)]
        # if "has_morph" means "has overlap with non-empty morf",
        # then the condition above is equivalent, since empty word_to_morphs => no morph.

        idx_to_pos = {idx: pos for pos, idx in enumerate(real_word_indices)}  # O(1)

        for i, w in enumerate(words_tier):
            word = w.mark.strip()
            if not word:
                continue

            morphs = word_to_morphs.get(i, [])
            morph = " | ".join(morphs)

            if not morph:
                continue

            pos = idx_to_pos.get(i)
            if pos is None:
                # in case of desynchronization — skip context
                continue

            left_idxs = real_word_indices[max(0, pos - 5):pos]
            right_idxs = real_word_indices[pos + 1:pos + 6]

            prev_words = [words_tier[j].mark.strip() for j in left_idxs]
            next_words = [words_tier[j].mark.strip() for j in right_idxs]
            context = " ".join(prev_words + [word] + next_words)

            rows.append({
                "conditions": condition,
                "word": word,
                "morph": morph,
                "context": context,
                "dialogue": dialogue_id,
                "ID": participant_id,
                "gender": gender,
                "year": year,
                "age": age,
                "education": education
            })


In [ ]:
dialogs_with_85 = set()

for row in rows:
    if row.get("age") == 85:
        dialog = str(row["dialogue"])[:3]
        dialogs_with_85.add(dialog)

rows = [
    row for row in rows
    if str(row["dialogue"])[:3] not in dialogs_with_85
]
any(r.get("age") == 85 for r in rows)

In [ ]:
df_rows = pd.DataFrame(rows)
print(df_rows)
df_sounds = pd.DataFrame(sounds)
#print(df_sounds)

In [ ]:
df_rows.to_csv("all_words_from_corpus.csv", index=False, encoding="utf-8")

In [ ]:
unique_by_gender = {}
for row in rows:
    gender = row["gender"]
    speaker = row["ID"]

    if gender not in unique_by_gender:
        unique_by_gender[gender] = set()

    unique_by_gender[gender].add(speaker)

for gender, speakers in unique_by_gender.items():
    print(gender, len(speakers))

In [ ]:
# Collect ages of unique speakers
speaker_ages = {}

for row in rows:
    speaker_id = row["ID"]
    age = row["age"]

    if speaker_id is not None and age is not None:
        speaker_ages[speaker_id] = age

ages = list(speaker_ages.values())

print("Unique speakers:", len(ages))
print("Minimum age:", min(ages))
print("Maximum age:", max(ages))
print("Median age:", statistics.median(ages))

In [ ]:
years = {row["year"] for row in rows if row["year"] is not None}

print(sorted(years))

In [ ]:
education = {row["education"] for row in rows if row["education"] is not None}
print(education)

In [ ]:
dialogue_genders = defaultdict(set)

for row in rows:
    dialogue = str(row["dialogue"])[:3]  
    gender = row["gender"]
    dialogue_genders[dialogue].add(gender)

mm_count = 0
ff_count = 0
mf_count = 0

for dial, genders in dialogue_genders.items():
    if genders == {1}:
        mm_count += 1
    elif genders == {0}:
        ff_count += 1
    else:
        mf_count += 1

print("M–M:", mm_count)
print("F–F:", ff_count)
print("M–F:", mf_count)

In [ ]:
dialogues = {str(row["dialogue"])[:3] for row in rows}
total_dialogues = len(dialogues)

print(total_dialogues)

In [ ]:
dialogue_ages = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    age = row["age"]

    if age is not None:
        dialogue_ages[dialog][speaker] = age

age_differences = []

for dialog, id_age_map in dialogue_ages.items():
    ages = list(id_age_map.values())

    if len(ages) == 2:
        diff = max(ages) - min(ages)
        age_differences.append(diff)

print("Number of dialogues:", len(age_differences))
print("Minimum age difference:", min(age_differences))
print("Maximum age difference:", max(age_differences))
print("Median age difference:", statistics.median(age_differences))

In [ ]:
dialogue_info = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    gender = row["gender"]
    age = row["age"]

    if age is not None:
        dialogue_info[dialog][speaker] = (gender, age)

men_older = 0
women_older = 0
same_age = 0

for dialog, speakers in dialogue_info.items():

    ages_m = [age for gender, age in speakers.values() if gender == 1]
    ages_f = [age for gender, age in speakers.values() if gender == 0]


    if ages_m and ages_f:
        m_age = min(ages_m)  
        f_age = min(ages_f)   
        if m_age > f_age:
            men_older += 1
        elif f_age > m_age:
            women_older += 1
        else:
            same_age += 1

print("Dialogues with an older male participant:", men_older)
print("Dialogues with an older female participant:", women_older)
print("Dialogues with same-age participants:", same_age)

In [ ]:
dialogue_ages = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    age = row["age"]
    if age is not None:
        dialogue_ages[dialog][speaker] = age


dialogue_to_diff = {}

for dialog, id_age_map in dialogue_ages.items():
    ages = list(id_age_map.values())
    if len(ages) >= 2:
        diff = max(ages) - min(ages)
    else:
        diff = None  
    dialogue_to_diff[dialog] = diff

for row in rows:
    dialog = str(row["dialogue"])[:3]
    row["age_diff"] = dialogue_to_diff.get(dialog)

In [ ]:
# 1. collect ages by dialogue 
dialogue_ages = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    age = row["age"]
    if age is not None:
        dialogue_ages[dialog][speaker] = age


# 2. compute age differences 
dialogue_to_age_diff = {}

for dialog, id_age_map in dialogue_ages.items():
    ages = list(id_age_map.values())
    if len(ages) >= 2:
        diff = max(ages) - min(ages)
        dialogue_to_age_diff[dialog] = diff
    else:
        dialogue_to_age_diff[dialog] = None


# 3. collect genders by dialogue
dialogue_genders = defaultdict(set)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    gender = row["gender"]
    if gender is not None:
        dialogue_genders[dialog].add(gender)


#  4. determine dialogue type by gender 
dialogue_to_speakers = defaultdict(dict)

for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]
    gender = row["gender"]
    if gender is not None:
        dialogue_to_speakers[dialog][speaker] = gender


# 5. add columns to rows: age_diff + interlocutor_gender
for row in rows:
    dialog = str(row["dialogue"])[:3]
    speaker = row["ID"]

    # age difference in this dialogue
    row["age_diff"] = dialogue_to_age_diff.get(dialog)

    # interlocutor gender (gender of the other participant in the same dialogue)
    speakers = dialogue_to_speakers.get(dialog, {})
    other_genders = [g for sid, g in speakers.items() if sid != speaker]
    row["second_speaker_gender"] = other_genders[0] if other_genders else None


In [ ]:
df_rows = pd.DataFrame(rows)
print(df_rows)

In [ ]:
counts = Counter((item['dialogue'], item['ID']) for item in rows)
for item in rows:
    item['speaker_word_count'] = counts[(item['dialogue'], item['ID'])]

In [ ]:
eitea = {}
eitea_counter = Counter()
for i, row in enumerate(rows):
    if not rows[i-1]['morph'].startswith('mina+0'):
        if row['word'] == 'ei' and rows[i+1]['word'] == 'tea':
            if rows[i+2]['word'] in ['ja', 'või', 'aga', 'ning']:
                eitea_counter[(row['dialogue'], row['ID'])] += 1
            elif not rows[i+2]['morph'].endswith('//_D_ //'):
                eitea_counter[(row['dialogue'], row['ID'])] += 1

    elif row['morph'].startswith('mina+0') and rows[i+1]['word'] == 'ei' and rows[i+2]['word'] == 'tea':
        if rows[i+3]['word'] in ['ja', 'või', 'aga', 'ning']:
            eitea_counter[(row['dialogue'], row['ID'])] += 1
        elif not rows[i+3]['morph'].endswith('//_D_ //'):
            eitea_counter[(row['dialogue'], row['ID'])] += 1
            
for i, row in enumerate(rows): 
    key = (row['dialogue'], row['ID'])
    
    if key not in eitea:
        eitea[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'eitea': eitea_counter[key]
        }
        
result = list(eitea.values())
df = pd.DataFrame(result)
df.to_csv('eitea.csv', index=False, encoding='utf-8')        

In [ ]:
with open('intensiivistajad.txt', encoding='utf-8') as f:
    intensifiers = f.read().split()

intensifiers_counter = Counter()
intensifiers_df = {}

# Initialize all speaker observations, including those with zero intensifiers
for row in rows:
    key = (row['dialogue'], row['ID'])
    if key not in intensifiers_df:
        intensifiers_df[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'intensifiers': 0
        }

# Count intensifiers followed by an adjective within the same speaker and dialogue
for i, row in enumerate(rows[:-1]):
    next_row = rows[i + 1]

    # Prevent counting across speaker or dialogue boundaries
    if (row['dialogue'], row['ID']) != (next_row['dialogue'], next_row['ID']):
        continue

    morph_parts = next_row['morph'].split()

    if row['word'] in intensifiers:
        if len(morph_parts) > 1 and morph_parts[1] == '//_A_':
            key = (row['dialogue'], row['ID'])
            intensifiers_counter[key] += 1

# Assign the final counts to all speaker observations
for key in intensifiers_df:
    intensifiers_df[key]['intensifiers'] = intensifiers_counter[key]

# Convert the results to a DataFrame and export to CSV
result = list(intensifiers_df.values())
df = pd.DataFrame(result)
df.to_csv('intensifiers.csv', index=False, encoding='utf-8')

# Verify the number of observations and intensifier counts
print("Observations:", len(df))
print("Total intensifiers:", df['intensifiers'].sum())
print("Zero observations:", (df['intensifiers'] == 0).sum())

In [ ]:
#kp_markers
kp_markers = Counter()

arvama = Counter()
uskuma = Counter()
kartma = Counter()
motlema = Counter()
utlema = Counter()

markers_df = {}

for i, row in enumerate(rows):
    
    if row['word'] in ['arvan', 'usun', 'eeldan', 'kardan', 'mõtlen', 'kaldun', 'ütlen']:
        kp_markers[(row['dialogue'], row['ID'])] += 1

    if row['word'] == 'arvan':
        arvama[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'usun':
        uskuma[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'kardan':
        kartma[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'mõtlen':
        motlema[(row['dialogue'], row['ID'])] += 1
    if row['word'] == 'ütlen':
        utlema[(row['dialogue'], row['ID'])] += 1

for i, row in enumerate(rows):
    key = (row['dialogue'], row['ID'])

    if key not in markers_df:
        markers_df[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'kp_markers': kp_markers[key],
            'arvama': arvama[key],
            'uskuma': uskuma[key],
            'kartma': kartma[key],
            'motlema': motlema[key],
            'utlema': utlema[key]
        }
        
result = list(markers_df.values())        
df = pd.DataFrame(result)
df.to_csv('kp_markers.csv', index=False, encoding='utf-8')  

In [ ]:
# particles
particle_counts = Counter()
jumal = Counter()
kurat = Counter()
polite_words = Counter()

# not_annotated_particles
certainty = Counter()
evaluation = Counter()
probability = Counter()
approximation = Counter()

skips = [
    'issand', 'jeerum', 'issake', 'issakene', 'kurat', 'raisk', 'vabandust',
    'aitäh', 'hops', 'kõll', 'pläraki', 'plaks', 'ata', 'tss', 'vaat',
    'mis_asja', 'oot', 'hahahah', 'hahah', 'hahaha', 'hah', 'hahaa',
    'hahaah', 'hahahahaa', 'ahah', 'ahaha', 'ahahah', 'ahahahah',
    'ähäh', 'äh', 'hee', 'heh', 'hehe', 'hehee', 'eheh', 'ehehe',
    'hihihii', 'hih', 'hii', 'ihi', 'hohoho', 'hohoo', 'hohoh',
    'hohohoo', 'huhuhuu', 'huh', 'huhh', 'uhoh', 'uhuhuu',
    'õhõh', 'hõh', 'õõh', 'õõhõ', 'uh', 'uhuhuu', 'uhoh',
    'oeh', 'ohh', 'oh', 'puh', 'puhh', 'juhhei'
]

for i, row in enumerate(rows):
    if (
        row['word'].startswith('jumal')
        and row['morph'].split('+', 1)[0].strip() == 'jumal'
    ):
        jumal[(row['dialogue'], row['ID'])] += 1
        # print(row['gender'], row['word'], row['morph'], row['context'])
        particle_counts[(row['dialogue'], row['ID'])] += 1

    if row['word'].startswith('kurad') or row['morph'].startswith(('sitt', 'pask')):
        # print(row['gender'], row['word'], row['morph'], row['context'])
        kurat[(row['dialogue'], row['ID'])] += 1
        particle_counts[(row['dialogue'], row['ID'])] += 1

    # not_annotated_particles
    if row['word'] in [
        'muidugi', 'tegelikult', 'kindlasti', 'tõesti', 'kahtlemata',
        'täiesti', 'just', 'tingimata', 'loomulikult',
        'paratamatult', 'päriselt', 'nagunii', 'niikuinii'
    ]:
        if rows[i + 1]['morph'].split()[1] != '//_A_':
            certainty[(row['dialogue'], row['ID'])] += 1
            particle_counts[(row['dialogue'], row['ID'])] += 1

    if row['word'] in ['üsna', 'nii', 'umbes', 'väga', 'eriti']:
        if rows[i + 1]['morph'].split()[1] != '//_A_':
            approximation[(row['dialogue'], row['ID'])] += 1
            particle_counts[(row['dialogue'], row['ID'])] += 1

    if row['word'] in ['kahjuks', 'õnneks', 'paraku']:
        evaluation[(row['dialogue'], row['ID'])] += 1
        particle_counts[(row['dialogue'], row['ID'])] += 1

    if row['word'] in [
        'vist', 'äkki', 'võibolla', 'tõenäoliselt',
        'ilmselt', 'põhimõtteliselt', 'üldiselt',
        'vast', 'arvatavasti'
    ]:
        probability[(row['dialogue'], row['ID'])] += 1
        particle_counts[(row['dialogue'], row['ID'])] += 1

    # politeness
    if row['word'] in [
        'palun', 'aitäh', 'vabandust', 'tere',
        'vabandage', 'vabanda', 
        'tervist', 'hallo', 'tsäu', 'tsäuki'
    ]:
        polite_words[(row['dialogue'], row['ID'])] += 1

    # particles
    if row['morph'].endswith('//_I_ //'):
        particle, morph = row['morph'].split('+')

        # if particle not in skips:
        #     particle_counts[(row['dialogue'], row['ID'])] += 1

        if particle in ['issand', 'jeerum', 'issake', 'issakene', 'jumel', 'jeesus']:
            jumal[(row['dialogue'], row['ID'])] += 1
            print(row['gender'], row['word'], row['morph'], row['context'])

        elif particle in ['kurat', 'raisk', 'lohh']:
            kurat[(row['dialogue'], row['ID'])] += 1
            print(row['gender'], row['word'], row['morph'], row['context'])

In [ ]:
new_data = {}

for row in rows:
    key = (row['dialogue'], row['ID'])

    if key not in new_data:
        new_data[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'polite_words': polite_words[key],
            'particle': particle_counts[key],
            'jumal': jumal[key],
            'kurat': kurat[key],
            'evaluation': evaluation[key],
            'certainty': certainty[key],
            'probability': probability[key],
            'approximation': approximation[key],
        }

# final list
result = list(new_data.values())

df = pd.DataFrame(result)
df.to_csv(
    'particles.csv',
    index=False,
    encoding='utf-8'
)

In [ ]:

RELIGIOUS_I = {'issand', 'jeerum', 'issake', 'issakene', 'jumel', 'jeesus'}
OBSCENE_I = {'kurat', 'raisk', 'lohh'}

# Reconstruct the original counting rules for both categories
by_speaker = defaultdict(lambda: {
    'religious_original': 0,
    'obscene_original': 0,
    'issand_contributions': 0,
    'kurat_contributions': 0,
    'issand_tokens': 0,
    'kurat_tokens': 0
})

lemma_composition_hits = Counter()
lemma_composition_tokens = Counter()
audit = []

for token_index, row in enumerate(rows):
    key = (row['dialogue'], row['ID'])
    word = str(row['word'])
    morph = str(row['morph'])
    lemma = morph.split('+', 1)[0].strip()

    is_particle_i = morph.endswith('//_I_ //')

    # Apply the same classification rules as in the original analysis
    religious_hits = int(
        word.startswith('jumal') and lemma == 'jumal'
    )
    obscene_hits = int(
        word.startswith('kurad') or
        morph.startswith(('sitt', 'pask'))
    )

    if is_particle_i:
        religious_hits += int(lemma in RELIGIOUS_I)
        obscene_hits += int(lemma in OBSCENE_I)

    # Remove only contributions belonging to the target lemmas
    issand_hits = religious_hits if lemma == 'issand' else 0
    kurat_hits = obscene_hits if lemma == 'kurat' else 0

    s = by_speaker[key]

    s['religious_original'] += religious_hits
    s['obscene_original'] += obscene_hits
    s['issand_contributions'] += issand_hits
    s['kurat_contributions'] += kurat_hits
    s['issand_tokens'] += int(issand_hits > 0)
    s['kurat_tokens'] += int(kurat_hits > 0)

    # Record lemma composition by gender
    if religious_hits:
        lemma_composition_hits[('religious', row['gender'], lemma)] += religious_hits
        lemma_composition_tokens[('religious', row['gender'], lemma)] += 1

    if obscene_hits:
        lemma_composition_hits[('obscene', row['gender'], lemma)] += obscene_hits
        lemma_composition_tokens[('obscene', row['gender'], lemma)] += 1

    # Flag surface forms that require manual inspection
    needs_review = word.startswith('kurad') and lemma != 'kurat'

    if religious_hits or obscene_hits or needs_review:
        audit.append({
            'token_index': token_index,
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'gender': row['gender'],
            'word': word,
            'lemma': lemma,
            'morph': morph,
            'context': row.get('context', ''),
            'religious_hits': religious_hits,
            'obscene_hits': obscene_hits,
            'issand_excluded_hits': issand_hits,
            'kurat_excluded_hits': kurat_hits,
            'review_kurad_surface': needs_review
        })

# Initialize all speaker observations, including zero counts
metadata = [
    'dialogue', 'ID', 'conditions', 'gender', 'year',
    'age', 'education', 'age_diff',
    'second_speaker_gender', 'speaker_word_count'
]

participants = {}

for row in rows:
    key = (row['dialogue'], row['ID'])

    if key not in participants:
        participants[key] = {
            col: row[col] for col in metadata
        }

result = []

for key, base in participants.items():
    s = by_speaker[key]

    # Verify that reconstructed counts match the original counters
    assert s['religious_original'] == jumal[key], (
        f"Religious category mismatch: {key}"
    )

    assert s['obscene_original'] == kurat[key], (
        f"Obscene category mismatch: {key}"
    )

    result.append({
        **base,
        'jumal': s['religious_original'],
        'kurat': s['obscene_original'],
        'issand_tokens': s['issand_tokens'],
        'kurat_lemma_tokens': s['kurat_tokens'],
        'jumal_without_issand': (
            s['religious_original'] - s['issand_contributions']
        ),
        'kurat_without_kurat': (
            s['obscene_original'] - s['kurat_contributions']
        )
    })

# Create and validate the sensitivity dataset
sensitivity_df = pd.DataFrame(result)

assert len(sensitivity_df) == 180
assert sensitivity_df[['dialogue', 'ID']].duplicated().sum() == 0

assert sensitivity_df[
    ['jumal_without_issand', 'kurat_without_kurat']
].ge(0).all().all()

# Verify agreement with the original particles.csv
original_particles = pd.read_csv(
    'particles.csv',
    dtype={'dialogue': str, 'ID': str}
)

comparison = sensitivity_df.merge(
    original_particles[['dialogue', 'ID', 'jumal', 'kurat']],
    on=['dialogue', 'ID'],
    how='outer',
    validate='one_to_one',
    suffixes=('_new', '_old'),
    indicator=True
)

assert (comparison['_merge'] == 'both').all()

for category in ['jumal', 'kurat']:
    assert (
        comparison[f'{category}_new'] ==
        comparison[f'{category}_old']
    ).all(), f"{category}: counts differ from particles.csv"

# Export sensitivity data
sensitivity_df.to_csv(
    'particles_sensitivity.csv',
    index=False,
    encoding='utf-8'
)

# Export token-level audit
pd.DataFrame(audit).to_csv(
    'particle_token_audit.csv',
    index=False,
    encoding='utf-8'
)

# Export lemma composition by gender
composition = pd.DataFrame([
    {
        'category': cat,
        'gender': gender,
        'lemma': lemma,
        'tokens': n,
        'category_hits': lemma_composition_hits[(cat, gender, lemma)]
    }
    for (cat, gender, lemma), n in lemma_composition_tokens.items()
]).sort_values(
    ['category', 'gender', 'tokens'],
    ascending=[True, True, False]
)

composition.to_csv(
    'particle_lemma_composition.csv',
    index=False,
    encoding='utf-8'
)

# Print summary
print("Speaker observations:", len(sensitivity_df))

print(
    "Original religious particles:",
    sensitivity_df['jumal'].sum()
)

print(
    "Issand tokens:",
    sensitivity_df['issand_tokens'].sum()
)

print(
    "Religious particles without issand:",
    sensitivity_df['jumal_without_issand'].sum()
)

print(
    "Original obscene particles:",
    sensitivity_df['kurat'].sum()
)

print(
    "Kurat tokens:",
    sensitivity_df['kurat_lemma_tokens'].sum()
)

print(
    "Obscene particles without kurat:",
    sensitivity_df['kurat_without_kurat'].sum()
)

print(
    "Surface-form exceptions to inspect:",
    sum(x['review_kurad_surface'] for x in audit)
)

print("Files saved successfully.")


In [ ]:
#pronounce
pronouns = Counter()
#patterns = ['sina', 'meie', 'teie']
sina_teie = Counter()
mina_meie = Counter()
mina = Counter()
for i, row in enumerate(rows):
    if row['morph'].startswith(('sina+', 'teie+')):
        pronouns[(row['dialogue'], row['ID'])] += 1
        sina_teie[(row['dialogue'], row['ID'])] += 1
    elif row['morph'].startswith(('mina+')):
        pronouns[(row['dialogue'], row['ID'])] += 1
        mina_meie[(row['dialogue'], row['ID'])] += 1
    elif row['morph'].startswith(('meie+')):
        pronouns[(row['dialogue'], row['ID'])] += 1
        mina_meie[(row['dialogue'], row['ID'])] += 1
        
markers_sina_teie = {}
for i, row in enumerate(rows):
    key = (row['dialogue'], row['ID'])
    
    if key not in markers_sina_teie:
        markers_sina_teie[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'pronouns_2face': sina_teie[key]
        }
        
result_sina_teie = list(markers_sina_teie.values())        
df_2face = pd.DataFrame(result_sina_teie)
df_2face.to_csv('pronouns_2face.csv', index=False, encoding='utf-8')  

markers_mina_meie = {}
for i, row in enumerate(rows):
    key = (row['dialogue'], row['ID'])
    
    if key not in markers_mina_meie:
        markers_mina_meie[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'pronouns_1face': mina_meie[key]
        }
        
result_mina_meie = list(markers_mina_meie.values())        
df_1face = pd.DataFrame(result_mina_meie)
df_1face.to_csv('pronouns_1face.csv', index=False, encoding='utf-8') 

markers = {}
for i, row in enumerate(rows):
    key = (row['dialogue'], row['ID'])
    
    if key not in markers:
        markers[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'pronouns': pronouns[key]
        }
        
results = list(markers.values())        
df_all = pd.DataFrame(results)
df_all.to_csv('pronouns_all.csv', index=False, encoding='utf-8') 

In [ ]:
reflexives = Counter()

reflexives_df = {}

for i, row in enumerate(rows):
    
    oma = row['morph'].split()
    if oma[0].startswith('oma') and oma[1].startswith('//_P_'):
        reflexives[(row['dialogue'], row['ID'])] += 1
       # print(row)
    elif oma[0].startswith('ise') and oma[1].startswith('//_P_'):
        reflexives[(row['dialogue'], row['ID'])] += 1

for i, row in enumerate(rows):
    key = (row['dialogue'], row['ID'])

    if key not in reflexives_df:
        reflexives_df[key] = {
            'dialogue': row['dialogue'],
            'ID': row['ID'],
            'conditions': row['conditions'],
            'gender': row['gender'],
            'year': row['year'],
            'age': row['age'],
            'education': row['education'],
            'age_diff': row['age_diff'],
            'second_speaker_gender': row['second_speaker_gender'],
            'speaker_word_count': row['speaker_word_count'],
            'reflexives': reflexives[key]
        }
#print(reflexives)
result = list(reflexives_df.values())  
print(result)
df = pd.DataFrame(result)
df.to_csv('reflexives.csv', index=False, encoding='utf-8')  